# Semantic Cache for LLM Responses

This notebook demonstrates using `SemanticCache` to reduce LLM API calls by caching responses.

**Benefits:**
- Reduces LLM costs by avoiding duplicate calls
- Lowers latency for repeated/similar queries
- **Two-level caching:** Level 1 (exact hash match) + Level 2 (semantic similarity)

**Built on:** RedisVL's SemanticCache + Redis Hash for exact matches

## Setup

Import dependencies and configure environment.

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Check if OpenAI API key is set
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY not found. Please set it in .env file or environment"
    )

# Redis connection
REDIS_URL = os.getenv("REDIS_URL", "redis://redis:6379")
print(f"Environment configured")
print(f"   Redis URL: {REDIS_URL}")

Environment configured
   Redis URL: redis://redis:6379


In [2]:
from agents import Agent, Runner

print("OpenAI Agents SDK imported successfully")

OpenAI Agents SDK imported successfully


## Create Semantic Cache

Initialize a semantic cache with configuration for similarity matching.

In [3]:
from redis_openai_agents import SemanticCache

cache = SemanticCache(
    redis_url=REDIS_URL,
    similarity_threshold=0.90,  # Match queries with 90%+ similarity
    ttl=3600,                    # Cache entries expire after 1 hour
    name="demo_llm_cache"        # Index name in Redis
)

print("SemanticCache created")
print(f"   Similarity threshold: {cache.similarity_threshold}")
print(f"   TTL: {cache.ttl} seconds")

10:36:13 sentence_transformers.SentenceTransformer INFO   Use pytorch device_name: cpu


10:36:13 sentence_transformers.SentenceTransformer INFO   Load pretrained SentenceTransformer: redis/langcache-embed-v1


SemanticCache created
   Similarity threshold: 0.9
   TTL: 3600 seconds


## Demo: Cache Miss (First Query)

The first time we ask a question, it's a cache miss - we need to call the LLM.

In [4]:
# Create a simple agent
agent = Agent(
    name="assistant",
    instructions="You are a helpful assistant. Be concise."
)

# First query
query = "What are the main benefits of using Redis?"

# Check cache first
cached = cache.get(query=query)
print(f"Cache hit: {cached is not None}")

if cached:
    print(f"\nCached response: {cached.response}")
else:
    print("\nCalling LLM...")
    result = await Runner.run(agent, input=query)
    response = result.final_output
    print(f"\nLLM Response: {response}")
    
    # Store in cache
    cache.set(
        query=query,
        response=response,
        metadata={"model": "gpt-4o-mini", "agent": agent.name}
    )
    print("\nStored in cache")

Cache hit: False

Calling LLM...


10:36:21 httpx INFO   HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"



LLM Response: The main benefits of using Redis are:

1. **High Performance:** Redis is extremely fast for both read and write operations due to its in-memory architecture.
2. **Low Latency:** Provides sub-millisecond response times, ideal for real-time applications.
3. **Versatile Data Structures:** Supports various data types such as strings, hashes, lists, sets, and sorted sets.
4. **Persistence Options:** Offers configurable persistence to disk (RDB snapshots, AOF logs) for data durability.
5. **Scalability:** Includes features like clustering, replication, and partitioning for horizontal scaling.
6. **Pub/Sub Messaging:** Supports real-time use cases with Publish/Subscribe messaging.
7. **Atomic Operations:** Provides atomic commands to ensure data consistency.
8. **Simplicity:** Simple to set up and use with straightforward APIs and commands.
9. **Community and Ecosystem:** Large community support and a rich ecosystem of tools, clients, and extensions.

These features make Redis 

## Demo: Cache Hit (Exact Match)

Asking the exact same question retrieves the cached response instantly.

In [5]:
# Same query again
cached = cache.get(query=query)
print(f"Cache hit: {cached is not None}")

if cached:
    print(f"Similarity: {cached.similarity:.2%}")
    print(f"\nCached response: {cached.response}")
    print(f"\nMetadata: {cached.metadata}")

Cache hit: True
Similarity: 100.00%

Cached response: The main benefits of using Redis are:

1. **High Performance:** Redis is extremely fast for both read and write operations due to its in-memory architecture.
2. **Low Latency:** Provides sub-millisecond response times, ideal for real-time applications.
3. **Versatile Data Structures:** Supports various data types such as strings, hashes, lists, sets, and sorted sets.
4. **Persistence Options:** Offers configurable persistence to disk (RDB snapshots, AOF logs) for data durability.
5. **Scalability:** Includes features like clustering, replication, and partitioning for horizontal scaling.
6. **Pub/Sub Messaging:** Supports real-time use cases with Publish/Subscribe messaging.
7. **Atomic Operations:** Provides atomic commands to ensure data consistency.
8. **Simplicity:** Simple to set up and use with straightforward APIs and commands.
9. **Community and Ecosystem:** Large community support and a rich ecosystem of tools, clients, and 

## Demo: Semantic Match (Similar Query)

The cache also matches semantically similar queries - a key feature that makes it more effective than simple string matching.

In [6]:
# Semantically similar query
similar_query = "Why should I use Redis?"
print(f"Original: {query}")
print(f"Similar:  {similar_query}\n")

cached = cache.get(query=similar_query)
print(f"Cache hit: {cached is not None}")

if cached:
    print(f"Similarity: {cached.similarity:.2%}")
    print(f"\nReturned cached response (no LLM call needed)")

Original: What are the main benefits of using Redis?
Similar:  Why should I use Redis?



Cache hit: True
Similarity: 93.64%

Returned cached response (no LLM call needed)


## Demo: Cache Miss (Different Topic)

Completely unrelated queries correctly return a cache miss.

In [7]:
# Completely different query
different_query = "How do I make a good pizza?"

cached = cache.get(query=different_query)
print(f"Query: {different_query}")
print(f"Cache hit: {cached is not None}")

if not cached:
    print("Correctly identified as unrelated - would need LLM call")

Query: How do I make a good pizza?
Cache hit: False
Correctly identified as unrelated - would need LLM call


## Two-Level Cache Architecture

The cache uses two levels for optimal performance:

- **Level 1 (L1):** O(1) Redis Hash lookup for exact string matches - fastest
- **Level 2 (L2):** Vector similarity search for semantic matches - handles paraphrases

L1 is checked first; only on L1 miss does L2 (more expensive) run.

In [8]:
# Demonstrate L1 vs L2 behavior
print("Creating fresh cache to demonstrate L1/L2...")
demo_cache = SemanticCache(
    redis_url=REDIS_URL,
    similarity_threshold=0.85,
    name="demo_l1_l2_cache"
)

# Store a query
demo_cache.set(query="What is machine learning?", response="ML is a subset of AI.")

# L1 hit - exact same query
result1 = demo_cache.get(query="What is machine learning?")
print(f"Exact query -> similarity: {result1.similarity:.2f} (L1 hit)")

# L2 hit - semantically similar
result2 = demo_cache.get(query="Explain machine learning to me")
if result2:
    print(f"Similar query -> similarity: {result2.similarity:.2f} (L2 hit)")

# Check stats
stats = demo_cache.get_stats()
print(f"\nL1 hits: {stats['l1_hits']}, L2 hits: {stats['l2_hits']}")

# Cleanup
demo_cache.clear()

Creating fresh cache to demonstrate L1/L2...
10:36:21 sentence_transformers.SentenceTransformer INFO   Use pytorch device_name: cpu


10:36:21 sentence_transformers.SentenceTransformer INFO   Load pretrained SentenceTransformer: redis/langcache-embed-v1


10:36:22 httpx INFO   HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"


Exact query -> similarity: 1.00 (L1 hit)

L1 hits: 1, L2 hits: 0
10:36:24 sentence_transformers.SentenceTransformer INFO   Use pytorch device_name: cpu


10:36:24 sentence_transformers.SentenceTransformer INFO   Load pretrained SentenceTransformer: redis/langcache-embed-v1


## Cache Statistics (L1/L2 Breakdown)

Track cache performance with built-in statistics, including Level 1 (exact) and Level 2 (semantic) hit breakdown.

In [9]:
stats = cache.get_stats()

print("Cache Statistics:")
print(f"   Total Hits: {stats['hits']}")
print(f"   Total Misses: {stats['misses']}")
print(f"   L1 Hits (exact match): {stats['l1_hits']}")
print(f"   L2 Hits (semantic match): {stats['l2_hits']}")

total = stats['hits'] + stats['misses']
if total > 0:
    hit_rate = stats['hits'] / total * 100
    print(f"   Hit Rate: {hit_rate:.1f}%")
    
if stats['hits'] > 0:
    l1_pct = stats['l1_hits'] / stats['hits'] * 100
    l2_pct = stats['l2_hits'] / stats['hits'] * 100
    print(f"\n   L1 vs L2 breakdown:")
    print(f"   - L1 (fast exact): {l1_pct:.1f}%")
    print(f"   - L2 (semantic): {l2_pct:.1f}%")

Cache Statistics:
   Total Hits: 2
   Total Misses: 2
   L1 Hits (exact match): 1
   L2 Hits (semantic match): 1
   Hit Rate: 50.0%

   L1 vs L2 breakdown:
   - L1 (fast exact): 50.0%
   - L2 (semantic): 50.0%


## Integration with Agent Runner

For convenience, you can wrap the cache check and LLM call together.

In [10]:
async def cached_agent_call(agent: Agent, query: str, cache: SemanticCache) -> str:
    """
    Call an agent with caching - checks cache first, stores result if miss.
    
    Args:
        agent: The agent to call
        query: User query
        cache: SemanticCache instance
        
    Returns:
        Response string (from cache or LLM)
    """
    # Check cache
    cached = cache.get(query=query)
    if cached:
        print(f"[CACHE HIT] similarity={cached.similarity:.2%}")
        return cached.response
    
    # Cache miss - call LLM
    print("[CACHE MISS] calling LLM...")
    result = await Runner.run(agent, input=query)
    response = result.final_output
    
    # Store in cache
    cache.set(
        query=query,
        response=response,
        metadata={"agent": agent.name}
    )
    
    return response

In [11]:
# Test the helper function
queries = [
    "What is Python?",
    "Tell me about Python programming",  # Should match first query semantically
    "What is JavaScript?",                # Different topic
    "What is Python?",                    # Exact match
]

for q in queries:
    print(f"\nQuery: {q}")
    response = await cached_agent_call(agent, q, cache)
    print(f"Response: {response[:80]}..." if len(response) > 80 else f"Response: {response}")


Query: What is Python?
[CACHE MISS] calling LLM...


10:36:27 httpx INFO   HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


Response: Python is a high-level, interpreted programming language known for its simplicit...

Query: Tell me about Python programming
[CACHE MISS] calling LLM...


10:36:28 httpx INFO   HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"


10:36:28 httpx INFO   HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"


10:36:32 httpx INFO   HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


Response: Python is a high-level, interpreted programming language known for its readabili...

Query: What is JavaScript?
[CACHE MISS] calling LLM...


10:36:33 httpx INFO   HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"


10:36:34 httpx INFO   HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


Response: JavaScript is a high-level, interpreted programming language commonly used to cr...

Query: What is Python?
[CACHE HIT] similarity=100.00%
Response: Python is a high-level, interpreted programming language known for its simplicit...


## Inspect Cache in Redis

Use RedisVL CLI to see what's stored in Redis.

In [12]:
!rvl index listall -u $REDIS_URL

10:36:34 [RedisVL] INFO   Using Redis address from environment variable, REDIS_URL
10:36:34 [RedisVL] INFO   Indices:
10:36:34 [RedisVL] INFO   1. demo_l1_l2_cache


10:36:34 [RedisVL] INFO   2. test_hybrid_filter
10:36:34 [RedisVL] INFO   3. demo_llm_cache
10:36:34 [RedisVL] INFO   4. knowledge_base


In [13]:
!rvl stats -i demo_llm_cache -u $REDIS_URL

10:36:35 [RedisVL] INFO   Using Redis address from environment variable, REDIS_URL

Statistics:
╭─────────────────────────────┬────────────╮
│ Stat Key                    │ Value      │
├─────────────────────────────┼────────────┤
│ num_docs                    │ 4          │
│ num_terms                   │ 311        │
│ max_doc_id                  │ 4          │
│ num_records                 │ 414        │
│ percent_indexed             │ 1          │
│ hash_indexing_failures      │ 0          │
│ number_of_uses              │ 7          │
│ bytes_per_record_avg        │ 79.0144958 │
│ doc_table_size_mb           │ 0.01588058 │
│ inverted_sz_mb              │ 0.03119659 │
│ key_table_size_mb           │ 1.65939331 │
│ offset_bits_per_record_avg  │ 8          │
│ offset_vectors_sz_mb        │ 4.51087951 │
│ offsets_per_term_avg        │ 1.14251208 │
│ records_per_doc_avg         │ 103.5      │
│ sortable_values_size_mb     │ 0          │
│ total_indexing_time         │ 0.81999999 │
│ to

## Clear Cache

Clear all cached entries when needed.

In [14]:
# Clear cache
cache.clear()
print("Cache cleared")

# Verify
cached = cache.get(query=query)
print(f"Previous query in cache: {cached is not None}")

10:36:35 sentence_transformers.SentenceTransformer INFO   Use pytorch device_name: cpu


10:36:35 sentence_transformers.SentenceTransformer INFO   Load pretrained SentenceTransformer: redis/langcache-embed-v1


Cache cleared
Previous query in cache: False


## Summary

This notebook demonstrated:

1. **SemanticCache Creation** - configure similarity threshold and TTL
2. **Cache Miss** - first query calls LLM and stores result
3. **Exact Match (L1)** - identical queries use fast O(1) hash lookup
4. **Semantic Match (L2)** - similar queries use vector similarity search
5. **L1/L2 Statistics** - track hit breakdown for optimization
6. **Integration** - wrap cache check with agent calls

**Key Benefits:**
- Reduce LLM costs (25%+ with good hit rate)
- Lower latency for repeated queries (L1 is O(1))
- Semantic matching catches paraphrased questions (L2)

**Next Steps:**
- Adjust `similarity_threshold` based on your use case
- Use TTL to control cache freshness
- Monitor L1 vs L2 hit rate and optimize accordingly